# SafeWall: XGBoost Runtime Prediction with Buffer Strategies
## Predicting HPC Job Runtimes to Prevent Premature Termination

---

### Problem Statement

HPC schedulers (e.g., PBS, Cobalt, Slurm) require users to declare a **walltime limit** before job submission.
If the actual runtime exceeds this limit, the scheduler terminates the job — wasting all compute time.
Users typically over-request walltime to stay safe, leading to poor cluster utilization.

**SafeWall** addresses this by training an XGBoost regression model on historical job data to predict
actual runtime. A configurable buffer is then added to the raw prediction, providing a safety margin
that reduces job termination risk while keeping the predicted walltime as tight as possible.

### Dataset

The dataset contains HPC jobs from the **Theta** supercomputer at Argonne National Laboratory,
covering a 7-year period (2017–2023). The scheduler is Cobalt. MaxNodes = 4,392.

### Sections

1. Data Loading and Exploration
2. Feature Engineering and Preprocessing
3. Train-Test Split
4. XGBoost Model Training with Hyperparameter Tuning
5. Model Evaluation
6. Feature Importance Analysis
7. Buffer Strategy Analysis
8. Results Summary

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
import xgboost as xgb
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
import joblib
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f'XGBoost version: {xgb.__version__}')

---
## Section 1: Data Loading and Exploration

The dataset is loaded from a preprocessed CSV derived from the Theta 7-year workload trace
(`data/theta_7year_worklog.swf`). Each row represents one completed HPC job. The target column is
`RUNTIME_SECONDS` — the actual wall-clock time the job ran before finishing.

Key engineered features include:
- **Previous runtimes** (`RUNTIME1`, `RUNTIME2`, `RUNTIME12`): the most recent past runtimes
  for the same user, which are strong predictors of future runtime.
- **Accuracy statistics** (`Amax`, `Aaverage`, `Tlongest`, etc.): per-user historical accuracy
  metrics describing how well the user's walltime requests have matched actual runtimes.
- **`cens`**: a censoring indicator used in Tobit-style models. In this dataset all jobs completed
  normally, so `cens` is always 0 and is excluded from features.

In [ ]:
df = pd.read_csv('data/theta_7year_dataset.csv')
print(f'Dataset shape: {df.shape}')
print(f'\nColumn names:\n{list(df.columns)}')
print(f'\nFirst few rows:')
df.head()

In [ ]:
# Summary statistics for the target variable
print('=== Target Variable: RUNTIME_SECONDS ===')
print(df['RUNTIME_SECONDS'].describe())
print(f'\nSkewness : {df["RUNTIME_SECONDS"].skew():.2f}')
print(f'Kurtosis : {df["RUNTIME_SECONDS"].kurtosis():.2f}')

# Censoring check — all jobs should be uncensored in this dataset
if 'cens' in df.columns:
    n_censored = df['cens'].sum()
    print(f'\nCensored jobs  : {n_censored} ({n_censored / len(df) * 100:.2f}%)')
    print(f'Uncensored jobs: {len(df) - n_censored} ({(len(df) - n_censored) / len(df) * 100:.2f}%)')

In [ ]:
# Missing value audit
print('=== Missing Values per Column ===')
missing = pd.DataFrame({
    'Missing Count': df.isnull().sum(),
    'Missing %': (df.isnull().sum() / len(df) * 100).round(2)
})
print(missing[missing['Missing Count'] > 0])

In [ ]:
# Runtime distribution (log scale for readability given high skew)
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].hist(df['RUNTIME_SECONDS'], bins=100, color='steelblue', edgecolor='black', alpha=0.7)
axes[0].set_xlabel('Runtime (seconds)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Runtime Distribution (Original Scale)')

axes[1].hist(np.log1p(df['RUNTIME_SECONDS']), bins=100, color='teal', edgecolor='black', alpha=0.7)
axes[1].set_xlabel('log(Runtime + 1)')
axes[1].set_ylabel('Frequency')
axes[1].set_title('Runtime Distribution (Log Scale)')

plt.tight_layout()
plt.show()

---
## Section 2: Feature Engineering and Preprocessing

The `user` and `group_id` columns are anonymized identifiers (hashed). They are retained
separately for the user-based buffer analysis in Section 7 but excluded from model features
to avoid data leakage and to keep the model generalizable to new users.

Rows containing any NaN values are dropped. The majority of missingness comes from
the accuracy-statistic columns, which are NaN for users whose first few jobs have no
historical data yet (~14.5% of records).

In [ ]:
# Define feature set — exclude identifiers, censoring flag, and target
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
exclude_cols = ['RUNTIME_SECONDS', 'user', 'group_id', 'cens']
feature_cols = [col for col in numeric_cols if col not in exclude_cols]

print(f'Model features ({len(feature_cols)} total):')
for i, col in enumerate(feature_cols, 1):
    print(f'  {i:2d}. {col}')

In [ ]:
X = df[feature_cols].copy()
y = df['RUNTIME_SECONDS'].copy()

# Preserve user column for personalized buffer computation (Section 7)
user_info = df[['user']].copy() if 'user' in df.columns else None

# Drop rows with any NaN — ensures clean training/test sets
valid_mask = X.notnull().all(axis=1)
X = X[valid_mask]
y = y[valid_mask]
if user_info is not None:
    user_info = user_info[valid_mask]

print(f'Rows before NaN removal : {valid_mask.shape[0]}')
print(f'Rows after NaN removal  : {len(X)}')
print(f'Rows removed            : {(~valid_mask).sum()}')

In [ ]:
# Pearson correlations between each feature and the target
correlations = (
    pd.concat([X, y], axis=1)
    .corr()['RUNTIME_SECONDS']
    .drop('RUNTIME_SECONDS')
    .sort_values(ascending=False)
)

print('=== Feature Correlations with RUNTIME_SECONDS ===')
print(correlations)

plt.figure(figsize=(10, 7))
correlations.plot(kind='barh', color='teal')
plt.axvline(x=0, color='black', linewidth=0.8)
plt.xlabel('Pearson Correlation Coefficient')
plt.title('Feature Correlations with RUNTIME_SECONDS', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap for the top 10 most correlated features
top_features = correlations.abs().nlargest(10).index.tolist() + ['RUNTIME_SECONDS']
corr_matrix = pd.concat([X, y], axis=1)[top_features].corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
    center=0, square=True, linewidths=0.5, cbar_kws={'shrink': 0.8}
)
plt.title('Correlation Heatmap — Top 10 Features + Target', fontweight='bold')
plt.tight_layout()
plt.show()

---
## Section 3: Train-Test Split

An 80/20 random split is used with `random_state=42` for reproducibility.
The user identity array is split alongside X and y so that user-level buffer statistics
can be computed on training jobs only and applied to test jobs.

In [ ]:
if user_info is not None:
    X_train, X_test, y_train, y_test, user_train, user_test = train_test_split(
        X, y, user_info, test_size=0.2, random_state=42
    )
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )
    user_train, user_test = None, None

print(f'Training set : {X_train.shape[0]:,} rows, {X_train.shape[1]} features')
print(f'Test set     : {X_test.shape[0]:,} rows, {X_test.shape[1]} features')

for label, series in [('Training', y_train), ('Test', y_test)]:
    print(f'\n{label} set — RUNTIME_SECONDS:')
    print(f'  Mean   : {series.mean():.2f}s')
    print(f'  Median : {series.median():.2f}s')
    print(f'  Std    : {series.std():.2f}s')
    print(f'  Min    : {series.min():.2f}s')
    print(f'  Max    : {series.max():.2f}s')

---
## Section 4: XGBoost Model Training with Hyperparameter Tuning

XGBoost (`XGBRegressor`) is well-suited to tabular regression problems with mixed feature
types and missing-value patterns. We use `RandomizedSearchCV` over 50 random parameter
combinations with 5-fold cross-validation, scoring by R².

The `tree_method='hist'` setting uses the histogram-based approximate split algorithm,
which dramatically speeds up training on large datasets.

In [ ]:
param_grid = {
    'n_estimators'     : [100, 200, 300, 400, 500],
    'learning_rate'    : [0.01, 0.05, 0.1, 0.15, 0.2],
    'max_depth'        : [3, 5, 7, 9, 11],
    'min_child_weight' : [1, 3, 5, 7],
    'subsample'        : [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree' : [0.7, 0.8, 0.9, 1.0],
    'gamma'            : [0, 0.1, 0.2, 0.3],
    'reg_alpha'        : [0, 0.01, 0.1, 1],
    'reg_lambda'       : [1, 1.5, 2, 3],
}

print('Hyperparameter search space:')
for param, values in param_grid.items():
    print(f'  {param:<20s}: {values}')

In [ ]:
search = RandomizedSearchCV(
    estimator=XGBRegressor(random_state=42, n_jobs=-1, tree_method='hist'),
    param_distributions=param_grid,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1,
    verbose=0,
)

print('Starting hyperparameter search (50 iterations, 5-fold CV)...')
search.fit(X_train, y_train)

print('\nBest hyperparameters:')
for param, value in search.best_params_.items():
    print(f'  {param:<20s}: {value}')
print(f'\nBest cross-validation R²: {search.best_score_:.6f}')

---
## Section 5: Model Evaluation

We report standard regression metrics on the held-out test set and quantify the
**underestimation rate** — the fraction of jobs for which the model predicts less time
than the job actually takes. In the walltime context, underestimation directly corresponds
to job termination risk before a buffer is applied.

In [ ]:
model = search.best_estimator_

y_train_pred = np.maximum(model.predict(X_train), 0)
y_test_pred  = np.maximum(model.predict(X_test),  0)

# Keep a clean copy of the base predictions for buffer comparisons
y_pred_base = y_test_pred.copy()

def regression_metrics(y_true, y_pred, label=''):
    """Print a standard set of regression metrics."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    mask = y_true > 0
    mape = np.mean(np.abs((np.asarray(y_true)[mask] - np.asarray(y_pred)[mask])
                          / np.asarray(y_true)[mask])) * 100
    print(f'  RMSE : {rmse:.2f}s')
    print(f'  MAE  : {mae:.2f}s')
    print(f'  R2   : {r2:.6f}')
    print(f'  MAPE : {mape:.2f}%')
    return rmse, mae, r2, mape

print('=== Training Set ===')
regression_metrics(y_train, y_train_pred)

print('\n=== Test Set ===')
test_rmse, test_mae, test_r2, test_mape = regression_metrics(y_test, y_test_pred)

# Underestimation analysis on test set
residuals_base = y_test.values - y_pred_base
n_under = (residuals_base > 0).sum()
print(f'\n=== Underestimation Analysis (Test Set, No Buffer) ===')
print(f'  Underestimations : {n_under:,} ({n_under / len(y_test) * 100:.2f}%)')
print(f'  Overestimations  : {len(y_test) - n_under:,} ({(len(y_test) - n_under) / len(y_test) * 100:.2f}%)')

In [ ]:
# Diagnostic plots: predictions vs actuals and residuals
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Scatter: original scale
axes[0, 0].scatter(y_test, y_pred_base, alpha=0.3, s=8, color='steelblue')
lims = [min(y_test.min(), y_pred_base.min()), max(y_test.max(), y_pred_base.max())]
axes[0, 0].plot(lims, lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0, 0].set_xlabel('Actual Runtime (s)')
axes[0, 0].set_ylabel('Predicted Runtime (s)')
axes[0, 0].set_title('Predictions vs Actuals (Original Scale)', fontweight='bold')
axes[0, 0].legend()

# Scatter: log scale
axes[0, 1].scatter(np.log1p(y_test), np.log1p(y_pred_base), alpha=0.3, s=8, color='seagreen')
log_lims = [np.log1p(y_test).min(), np.log1p(y_test).max()]
axes[0, 1].plot(log_lims, log_lims, 'r--', lw=1.5, label='Perfect prediction')
axes[0, 1].set_xlabel('log(Actual Runtime + 1)')
axes[0, 1].set_ylabel('log(Predicted Runtime + 1)')
axes[0, 1].set_title('Predictions vs Actuals (Log Scale)', fontweight='bold')
axes[0, 1].legend()

# Residual plot
residuals = y_test.values - y_pred_base
axes[1, 0].scatter(y_pred_base, residuals, alpha=0.3, s=8, color='mediumpurple')
axes[1, 0].axhline(0, color='red', linestyle='--', lw=1.5)
axes[1, 0].set_xlabel('Predicted Runtime (s)')
axes[1, 0].set_ylabel('Residual (Actual - Predicted)')
axes[1, 0].set_title('Residual Plot', fontweight='bold')

# Residuals histogram
axes[1, 1].hist(residuals, bins=100, color='coral', edgecolor='black', alpha=0.7)
axes[1, 1].axvline(residuals.mean(),   color='red',   linestyle='--', lw=1.5,
                   label=f'Mean: {residuals.mean():.1f}s')
axes[1, 1].axvline(np.median(residuals), color='navy', linestyle='--', lw=1.5,
                   label=f'Median: {np.median(residuals):.1f}s')
axes[1, 1].set_xlabel('Residual (s)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Residuals Distribution', fontweight='bold')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Absolute and percentage error statistics
abs_errors = np.abs(residuals)
mask = y_test.values > 0
pct_errors = ((y_test.values[mask] - y_pred_base[mask]) / y_test.values[mask]) * 100
pct_errors = pct_errors[np.isfinite(pct_errors)]

print('=== Absolute Error Statistics ===')
for label, val in [
    ('Mean',   abs_errors.mean()),
    ('Median', np.median(abs_errors)),
    ('Std',    abs_errors.std()),
    ('P75',    np.percentile(abs_errors, 75)),
    ('P90',    np.percentile(abs_errors, 90)),
    ('P95',    np.percentile(abs_errors, 95)),
    ('P99',    np.percentile(abs_errors, 99)),
]:
    print(f'  {label:<8s}: {val:.2f}s')

print('\n=== Percentage Error Statistics ===')
print(f'  Mean   : {pct_errors.mean():.2f}%')
print(f'  Median : {np.median(pct_errors):.2f}%')
print(f'  Std    : {pct_errors.std():.2f}%')

---
## Section 6: Feature Importance Analysis

XGBoost provides feature importances based on how often each feature is used to split
nodes (weight), the average gain from splits using each feature (gain), and the fraction
of training samples covered by splits on each feature (cover).

We plot both weight-based importance (from `feature_importances_`) and gain-based
importance to cross-check which features matter most.

In [ ]:
feature_importance = (
    pd.DataFrame({'feature': feature_cols, 'importance': model.feature_importances_})
    .sort_values('importance', ascending=False)
    .reset_index(drop=True)
)

print('=== Feature Importance (Weight) ===')
print(feature_importance.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

top_n = 15
axes[0].barh(
    range(top_n),
    feature_importance['importance'].head(top_n),
    color='steelblue'
)
axes[0].set_yticks(range(top_n))
axes[0].set_yticklabels(feature_importance['feature'].head(top_n))
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance Score')
axes[0].set_title(f'Top {top_n} Features (Weight)', fontweight='bold')

xgb.plot_importance(model, max_num_features=top_n, importance_type='gain', ax=axes[1])
axes[1].set_title(f'Top {top_n} Features (Gain)', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Section 7: Buffer Strategy Analysis

The base model achieves good accuracy but still underestimates the runtime of about
48% of jobs. Adding a **buffer** to every prediction trades some overestimation for
a reduction in the job-killing underestimation rate.

We test five families of buffer strategies:

| Strategy | Description |
|---|---|
| 1. Percentage-based | `pred * (1 + p)` for p in {5%, 10%, 15%, 20%, 25%, 30%} |
| 2. Fixed value | `pred + c` for c in {30, 60, 120, 180, 300, 600} seconds |
| 3. Adaptive | Smaller % for short jobs, larger % for long jobs (threshold: 3600 s) |
| 4. Error-based | `pred + percentile(training_errors, p)` — buffer derived from training residuals |
| 5. User-based | Personalized buffer per user from their own error history (full or rolling 5-job window) |

The evaluation helper below computes all key metrics for a given set of buffered predictions.

In [ ]:
def evaluate_buffer(predictions, y_true):
    """
    Evaluate the performance of a buffered prediction strategy.

    Parameters
    ----------
    predictions : np.ndarray
        Buffered runtime predictions in seconds.
    y_true : np.ndarray or pd.Series
        Actual runtime values in seconds.

    Returns
    -------
    dict
        Dictionary containing RMSE, MAE, R2, accuracy, underestimation rate,
        and mean/median residuals.
    """
    y_true = np.asarray(y_true)
    residuals = y_true - predictions
    n_under = (residuals > 0).sum()

    # Accuracy: min(pred, actual) / max(pred, actual), symmetric overlap ratio
    accuracy = np.mean(
        np.minimum(predictions, y_true) / np.maximum(predictions, y_true)
    )

    return {
        'rmse'              : np.sqrt(mean_squared_error(y_true, predictions)),
        'mae'               : mean_absolute_error(y_true, predictions),
        'r2'                : r2_score(y_true, predictions),
        'accuracy'          : accuracy,
        'underestimations'  : n_under,
        'underest_pct'      : n_under / len(y_true) * 100,
        'mean_residual'     : residuals.mean(),
        'median_residual'   : np.median(residuals),
    }

# Accumulate all results and save buffered predictions
buffer_results = {}
result_df = pd.DataFrame({
    'True_Runtime'    : y_test.values,
    'User_Walltime'   : X_test['WALLTIME_SECONDS'].values,
    'Predicted'       : y_pred_base,
})

### Strategy 1: Percentage-Based Buffers

The simplest approach: scale every prediction up by a fixed percentage.
A 5% buffer on a 1-hour prediction adds 3 minutes; the same 5% on a 10-hour job adds 30 minutes.
The buffer grows with predicted duration, which is intuitive but can produce large absolute
overestimates for very long jobs.

In [ ]:
print('Strategy 1: Percentage-Based Buffers')
print('-' * 60)

for pct in [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]:
    buffered = y_pred_base * (1 + pct)
    key = f'Pct_{int(pct * 100)}pct'
    result = evaluate_buffer(buffered, y_test)
    buffer_results[key] = result
    result_df[f'Buffer_Pct{int(pct*100)}pct'] = buffered
    print(
        f'  +{int(pct*100):2d}%  RMSE={result["rmse"]:8.2f}s  '
        f'Underest={result["underest_pct"]:5.2f}%  '
        f'Mean residual={result["mean_residual"]:9.2f}s'
    )

### Strategy 2: Fixed Value Buffers

Add a constant number of seconds to every prediction regardless of job duration.
This provides a uniform safety margin and tends to help short jobs (where even a small
underestimate is proportionally large) without massively inflating walltime for long jobs.

In [ ]:
print('Strategy 2: Fixed Value Buffers')
print('-' * 60)

for seconds in [30, 60, 120, 180, 300, 600]:
    buffered = y_pred_base + seconds
    key = f'Fixed_{seconds}s'
    result = evaluate_buffer(buffered, y_test)
    buffer_results[key] = result
    result_df[f'Buffer_Fixed{seconds}s'] = buffered
    print(
        f'  +{seconds:4d}s  RMSE={result["rmse"]:8.2f}s  '
        f'Underest={result["underest_pct"]:5.2f}%  '
        f'Mean residual={result["mean_residual"]:9.2f}s'
    )

### Strategy 3: Adaptive Buffers

Apply a smaller percentage buffer to short jobs and a larger percentage to long jobs,
using a threshold of 3600 seconds (1 hour). The motivation is that long jobs have
intrinsically higher absolute uncertainty, so they benefit from a proportionally
larger safety margin.

In [ ]:
print('Strategy 3: Adaptive Buffers (threshold = 3600 s)')
print('-' * 60)

THRESHOLD = 3600  # 1 hour

adaptive_configs = [
    {'name': 'Adaptive_5-10pct',  'short': 0.05, 'long': 0.10},
    {'name': 'Adaptive_10-15pct', 'short': 0.10, 'long': 0.15},
    {'name': 'Adaptive_10-20pct', 'short': 0.10, 'long': 0.20},
]

for cfg in adaptive_configs:
    buffered = y_pred_base.copy()
    short_mask = y_pred_base <= THRESHOLD
    long_mask  = y_pred_base >  THRESHOLD
    buffered[short_mask] *= (1 + cfg['short'])
    buffered[long_mask]  *= (1 + cfg['long'])

    result = evaluate_buffer(buffered, y_test)
    buffer_results[cfg['name']] = result
    result_df[f'Buffer_{cfg["name"]}'] = buffered
    print(
        f'  {cfg["name"]:<22s}  RMSE={result["rmse"]:8.2f}s  '
        f'Underest={result["underest_pct"]:5.2f}%  '
        f'Mean residual={result["mean_residual"]:9.2f}s'
    )

### Strategy 4: Error-Based Buffers

Use the distribution of absolute prediction errors on the **training set** to choose
the buffer size. Adding the P90 training error as a fixed offset means the model
would have correctly bounded 90% of training jobs. This is a data-driven approach
that grounds the buffer in observed model uncertainty.

In [ ]:
# Compute absolute errors on the training set
train_abs_errors = np.abs(y_train.values - y_train_pred)

print('Training set absolute error percentiles:')
for p in [50, 75, 90, 95]:
    print(f'  P{p:2d}: {np.percentile(train_abs_errors, p):.2f}s')

print('\nStrategy 4: Error-Based Buffers')
print('-' * 60)

for p in [50, 75, 90, 95]:
    buffer_val = np.percentile(train_abs_errors, p)
    buffered   = y_pred_base + buffer_val
    key        = f'ErrorP{p}'
    result = evaluate_buffer(buffered, y_test)
    buffer_results[key] = result
    result_df[f'Buffer_ErrorP{p}'] = buffered
    print(
        f'  P{p:2d} (+{buffer_val:7.2f}s)  RMSE={result["rmse"]:8.2f}s  '
        f'Underest={result["underest_pct"]:5.2f}%  '
        f'Mean residual={result["mean_residual"]:9.2f}s'
    )

### Strategy 5: User-Based Buffers (Personalized)

Different users have systematically different prediction error patterns — some users
always run shorter-than-predicted jobs, others consistently exceed predictions.
A personalized buffer derived from each user's own error history can be more precise
than a global buffer.

We test two variants:

- **Full History**: uses all training-set predictions for each user to compute their
  error percentile. Stable but slow to adapt to behavioral change.
- **Rolling Window (last 5 jobs)**: uses only the user's 5 most recent training-set
  jobs. More responsive to recent behavior but noisier for users with few jobs.

Users with fewer than 3 historical jobs fall back to the global error-percentile buffer.

In [ ]:
if user_train is None or user_test is None:
    print('User information not available. Skipping user-based buffers.')
else:
    # Build a training-set frame with per-job absolute errors and job sequence numbers
    train_err_df = pd.DataFrame({
        'user'     : user_train['user'].values,
        'actual'   : y_train.values,
        'predicted': y_train_pred,
        'abs_error': train_abs_errors,
    })
    train_err_df['job_seq'] = train_err_df.groupby('user').cumcount()

    print(f'Training jobs  : {len(train_err_df):,}')
    print(f'Unique users   : {train_err_df["user"].nunique():,}')
    print(f'Jobs/user mean : {train_err_df.groupby("user").size().mean():.1f}')
    print(f'Jobs/user med  : {train_err_df.groupby("user").size().median():.0f}')

In [ ]:
if user_train is not None and user_test is not None:

    def compute_user_buffers_full(group):
        """
        Compute buffer percentiles from a user's complete training history.

        Uses all historical absolute prediction errors for the user.
        Returns NaN for all percentiles when the user has fewer than 3 jobs,
        signaling that the global fallback should be used.

        Parameters
        ----------
        group : pd.DataFrame
            Subset of train_err_df for one user, containing 'abs_error'.

        Returns
        -------
        pd.Series
            Per-user buffer values at P50, P75, P90, P95 and job count.
        """
        errors = group['abs_error']
        if len(errors) < 3:
            return pd.Series({'p50': np.nan, 'p75': np.nan,
                              'p90': np.nan, 'p95': np.nan, 'n_jobs': len(errors)})
        return pd.Series({
            'p50'   : np.percentile(errors, 50),
            'p75'   : np.percentile(errors, 75),
            'p90'   : np.percentile(errors, 90),
            'p95'   : np.percentile(errors, 95),
            'n_jobs': len(errors),
        })

    def compute_user_buffers_rolling(group, window=5):
        """
        Compute buffer percentiles from a user's most recent training jobs.

        Selects the `window` most recent jobs by job sequence number and
        computes error percentiles from those jobs only. This adapts quickly
        to recent changes in a user's job patterns.

        Parameters
        ----------
        group : pd.DataFrame
            Subset of train_err_df for one user.
        window : int
            Number of recent jobs to use (default: 5).

        Returns
        -------
        pd.Series
            Per-user buffer values at P50, P75, P90, P95 and recent job count.
        """
        recent_errors = group.nlargest(window, 'job_seq')['abs_error']
        if len(recent_errors) < 3:
            return pd.Series({'p50': np.nan, 'p75': np.nan,
                              'p90': np.nan, 'p95': np.nan, 'n_jobs': len(recent_errors)})
        return pd.Series({
            'p50'   : np.percentile(recent_errors, 50),
            'p75'   : np.percentile(recent_errors, 75),
            'p90'   : np.percentile(recent_errors, 90),
            'p95'   : np.percentile(recent_errors, 95),
            'n_jobs': len(recent_errors),
        })

    user_buf_full    = train_err_df.groupby('user').apply(compute_user_buffers_full).reset_index()
    user_buf_rolling = train_err_df.groupby('user').apply(
        lambda g: compute_user_buffers_rolling(g, window=5)
    ).reset_index()

    print('Full history   — users with >= 3 jobs :',
          int((user_buf_full['n_jobs'] >= 3).sum()))
    print('Rolling window — users with >= 3 jobs :',
          int((user_buf_rolling['n_jobs'] >= 3).sum()))

In [ ]:
if user_train is not None and user_test is not None:

    # Global fallback values derived from training error percentiles
    global_fallback = {
        'p50': np.percentile(train_abs_errors, 50),
        'p75': np.percentile(train_abs_errors, 75),
        'p90': np.percentile(train_abs_errors, 90),
        'p95': np.percentile(train_abs_errors, 95),
    }
    print('Global fallback buffers:', {k: f'{v:.2f}s' for k, v in global_fallback.items()})

    test_base_df = pd.DataFrame({
        'user'      : user_test['user'].values,
        'base_pred' : y_pred_base,
    })

    for variant_name, user_buf_df in [
        ('UserFull',    user_buf_full),
        ('UserRoll5',   user_buf_rolling),
    ]:
        print(f'\nVariant: {variant_name}')
        print('-' * 60)
        for pct_key in ['p50', 'p75', 'p90', 'p95']:
            merged = test_base_df.merge(user_buf_df[['user', pct_key]], on='user', how='left')
            merged[pct_key] = merged[pct_key].fillna(global_fallback[pct_key])
            buffered = (merged['base_pred'] + merged[pct_key]).values

            key = f'{variant_name}_{pct_key.upper()}'
            result = evaluate_buffer(buffered, y_test)
            buffer_results[key] = result
            result_df[f'Buffer_{key}'] = buffered

            print(
                f'  {pct_key.upper()}  RMSE={result["rmse"]:8.2f}s  '
                f'Underest={result["underest_pct"]:5.2f}%  '
                f'Mean residual={result["mean_residual"]:9.2f}s'
            )

---
## Section 8: Results Summary

The table below consolidates results for all strategies. Key metrics to balance:

- **Underestimation %** — the primary metric from a job-safety perspective. Lower means
  fewer jobs are at risk of being killed before they finish.
- **RMSE** — measures total prediction error including overestimation. A higher buffer
  always increases RMSE; the goal is to find the best underestimation reduction per
  unit of RMSE increase.
- **Mean Residual** — a negative value means the buffered prediction overestimates on
  average. The magnitude indicates how much walltime is 'wasted' on average.

In [ ]:
# Baseline row (no buffer)
baseline_result = evaluate_buffer(y_pred_base, y_test)

rows = [{
    'Strategy'         : 'Baseline (no buffer)',
    'RMSE (s)'         : baseline_result['rmse'],
    'MAE (s)'          : baseline_result['mae'],
    'R2'               : baseline_result['r2'],
    'Accuracy'         : baseline_result['accuracy'],
    'Underest %'       : baseline_result['underest_pct'],
    'Mean Residual (s)': baseline_result['mean_residual'],
}]

for name, result in buffer_results.items():
    rows.append({
        'Strategy'         : name,
        'RMSE (s)'         : result['rmse'],
        'MAE (s)'          : result['mae'],
        'R2'               : result['r2'],
        'Accuracy'         : result['accuracy'],
        'Underest %'       : result['underest_pct'],
        'Mean Residual (s)': result['mean_residual'],
    })

summary_df = pd.DataFrame(rows)
summary_df = summary_df.round({'RMSE (s)': 2, 'MAE (s)': 2, 'R2': 6,
                               'Accuracy': 4, 'Underest %': 2, 'Mean Residual (s)': 2})

pd.set_option('display.max_rows', 60)
pd.set_option('display.width', 120)
print(summary_df.to_string(index=False))

In [ ]:
# Visualize the underestimation vs RMSE trade-off across all strategies
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Underestimation rate for all strategies
axes[0].barh(
    summary_df['Strategy'],
    summary_df['Underest %'],
    color='salmon', edgecolor='black', alpha=0.8
)
axes[0].axvline(x=5, color='red', linestyle='--', lw=1.5, label='5% target')
axes[0].set_xlabel('Underestimation Rate (%)')
axes[0].set_title('Underestimation Rate by Strategy', fontweight='bold')
axes[0].invert_yaxis()
axes[0].legend()

# RMSE for all strategies
axes[1].barh(
    summary_df['Strategy'],
    summary_df['RMSE (s)'],
    color='steelblue', edgecolor='black', alpha=0.8
)
axes[1].axvline(x=baseline_result['rmse'], color='red', linestyle='--', lw=1.5,
                label='Baseline RMSE')
axes[1].set_xlabel('RMSE (seconds)')
axes[1].set_title('RMSE by Strategy', fontweight='bold')
axes[1].invert_yaxis()
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Scatter: RMSE vs underestimation rate — the core trade-off
fig, ax = plt.subplots(figsize=(12, 7))

ax.scatter(
    summary_df['Underest %'],
    summary_df['RMSE (s)'],
    s=60, color='steelblue', zorder=5
)
for _, row in summary_df.iterrows():
    ax.annotate(
        row['Strategy'],
        (row['Underest %'], row['RMSE (s)']),
        fontsize=7, xytext=(4, 4), textcoords='offset points'
    )

ax.set_xlabel('Underestimation Rate (%)')
ax.set_ylabel('RMSE (seconds)')
ax.set_title('Trade-off: Underestimation Rate vs RMSE', fontweight='bold')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Highlight the best strategies for the target of <= 5% underestimation
TARGET_UNDEREST = 5.0

candidates = summary_df[summary_df['Underest %'] <= TARGET_UNDEREST].copy()
candidates = candidates.sort_values('RMSE (s)')

print(f'Strategies achieving underestimation <= {TARGET_UNDEREST}%,')
print(f'sorted by RMSE (lower is better):')
print()
print(candidates[['Strategy', 'RMSE (s)', 'MAE (s)', 'Underest %', 'Mean Residual (s)']]
      .to_string(index=False))

if not candidates.empty:
    best = candidates.iloc[0]
    print(f'\nRecommended strategy : {best["Strategy"]}')
    print(f'  RMSE             : {best["RMSE (s)"]:.2f}s')
    print(f'  Underestimation  : {best["Underest %"]:.2f}%')
    print(f'  Mean residual    : {best["Mean Residual (s)"]:.2f}s')

In [ ]:
# Print a concise summary of the overall results
print('=' * 60)
print('SAFEWALL — RESULTS SUMMARY')
print('=' * 60)
print()
print('Base model performance (test set):')
print(f'  R2   : {test_r2:.4f}')
print(f'  RMSE : {test_rmse:.0f}s')
print(f'  MAPE : {test_mape:.2f}%')
print()
print('Without buffer:')
print(f'  Underestimation rate : {n_under / len(y_test) * 100:.2f}%')
print()
print('Best fixed-buffer strategy (lowest RMSE under 5% underestimation):')
if not candidates.empty:
    best = candidates.iloc[0]
    print(f'  Strategy             : {best["Strategy"]}')
    print(f'  RMSE                 : {best["RMSE (s)"]:.2f}s')
    print(f'  Underestimation rate : {best["Underest %"]:.2f}%')
    print(f'  Mean residual        : {best["Mean Residual (s)"]:.2f}s')
print()
print('Output saved to: data/predictions_with_buffers.csv')

In [ ]:
# Save all predictions and buffer variants for downstream analysis
result_df.to_csv('data/predictions_with_buffers.csv', index=False)
print(f'Saved {len(result_df):,} rows to data/predictions_with_buffers.csv')
print(f'Columns: {list(result_df.columns)}')